In [1]:
import os

os.chdir("..")

(chapter-indexing)=
Slicing & Dicing Data with pandas
=================================

:::{admonition} Learning Objectives
* Explain what pandas indexes are and how they're used
* Get, modify, and set indexes on series and data frames
* Explain and use pandas' index alignment
* Describe and use the 4 major modes of indexing in pandas
* Explain the difference between `.loc` and `.at`
* Identify problems where using `.set_index` to make a column an index is
  helpful
* Explain and use pandas' query mini-language
:::

This chapter is a deep dive into how **indexing**&mdash;the process of getting
or setting elements of a data structure&mdash;works in the [pandas][] package.
Indexing is sometimes also called **subsetting** or (element) extraction, and
is a fundamental operation when using a programming language to solve problems
in data science.

[pandas]: https://pandas.pydata.org/


Prerequisites
-------------

This chapter assumes you already have basic familiarity with Python and pandas.
In particular, you should be comfortable indexing Python lists and dictionaries
with the square bracket operator `[ ]`, and should be able to explain what a
`Series` and `DataFrame` are and how the they differ. DataLab's [Python Basics
Reader][py-basics] and its accompanying workshop provide a suitable
introduction to these topics.

[py-basics]: https://ucdavisdatalab.github.io/workshop_python_basics/

To follow along, you'll need the following software versions (or newer)
installed on your computer:

* [Python][] 3.10
* [NumPy][] 1.22
* [pandas][] 1.4

One way to install these is to install the [Anaconda][] Python distribution.
Chapter 2 provides more details about Anaconda and the `conda` package manager.

[Python]: https://www.python.org/
[NumPy]: https://numpy.org/
[Anaconda]: https://www.anaconda.com/

[CLICK HERE][data-download] to get the data set used in the examples for this
chapter. After clicking the link, you'll need to right-click on the page and
select "Save Page As...". Make sure to save the page somewhere you can easily
access from your Python session.

[data-download]: https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2022/2022-03-29/sports.csv


What's an Index?
----------------

In pandas, an `Index` is a Python object that stores a sequence of labels or
names. The labels usually correspond to the elements of a data structure along
a given dimension or **axis**. Indexes are a key feature that differentiates
pandas from other software for programming with tabular data (such as R).

According to the pandas documentation, indexes serve three important roles:

1. As metadata to provide additional context about a data set
2. As a way to explicitly and automatically align data, avoiding bugs due to
   misalignment (see {numref}`alignment`)
3. As a convenience for getting and setting subsets of the data (see
   {numref}`column-indexes`)


### Indexes on Series

As an example of where pandas uses indexes, `Series` are 1-dimensional and
always have exactly one index. You can see the index just to the left of the
elements when you print out a series:

In [2]:
import pandas as pd

x = pd.Series([4, 5, 3, 1])
x

0    4
1    5
2    3
3    1
dtype: int64

You can get or set the index on a series with the `.index` attribute:

In [3]:
x.index

RangeIndex(start=0, stop=4, step=1)

The labels in an index can be numbers, strings, dates, or other values. Pandas
provides subclasses of `Index` for specific purposes, including:

* `RangeIndex`, where the labels are a monotonic sequence of integers with a
  fixed beginning, end, and step size
* `CategoricalIndex`, where the labels are categories (and not necessarily
  unique)
* `IntervalIndex`, where the labels are non-overlapping intervals of the real
  number line
* `DateTimeIndex`, where the labels are dates and times
* `PeriodIndex`, where each label is a date and time span

You may also encounter `Int64Index`, `UInt64Index`, and `Float64Index`, which
represent an arbitrary list of labels of the respective type. For example:

In [4]:
x[[1, 3]].index

Int64Index([1, 3], dtype='int64')

Beginning in pandas 2.0, these three subclasses (but not subclasses in the
bulleted list above) will be replaced with a generic `Index` class. See [this
warning][index-deprecated] in the pandas documentation for details.

[index-deprecated]: https://pandas.pydata.org/docs/user_guide/advanced.html#int64index-and-rangeindex

You can access the labels in an index with standard Python indexing operations:

In [5]:
idx = x.index
# Get the 2nd label
idx[1]

1

Indexes are **immutable**, which means the labels in an index can't be changed.
For instance:

In [6]:
idx[0] = 5

TypeError: Index does not support mutable operations

Although labels in an index can't be changed, every index also has a name,
which can be changed. Think of the name as a name for the dimension or axis to
which the index corresponds. You can get or set the name of an index with the
`.name` attribute (on the index itself):

In [7]:
idx.name

The default value of `.name` is `None`, which means the index doesn't have a
name. When pandas prints a series or data frame, it will also print the names
of any attached indexes. For example:

In [8]:
x.index.name = "labels"
x

labels
0    4
1    5
2    3
3    1
dtype: int64

Pandas automatically creates indexes whenever you create (or load) a series or
data frame. Pandas also provides functions to construct indexes manually. These
are usually named after the type of index. For instance, you can create an
ordinary `Index` from a list or NumPy array:

In [9]:
new_idx = pd.Index(["a", "b", "c", "d"])
new_idx

Index(['a', 'b', 'c', 'd'], dtype='object')

You can use assignment to replace the index on a series. For example:

In [10]:
x.index = new_idx
x

a    4
b    5
c    3
d    1
dtype: int64

Note that the new index must have the same length as the index it replaces (and
the series itself).

:::{admonition} Checkpoint Exercise
:class: important

Replace the index on `x` with one like the original (in variable `idx`), but
where the first label is 5 instead of 0. Try to do this *without* typing out a
list of all 4 labels.

Hint: the Python `list` function can convert many types of objects into lists,
including pandas indexes.
:::


### Indexes on Data Frames

Pandas `DataFrame`s are 2-dimensional and always have exactly two indexes. The
indexes correspond to the rows and the columns, respectively. To see this, use
pandas to load the data from the `sports.csv` file:

In [11]:
dtype = {"classification_other": str}
sports = pd.read_csv("data/sports.csv", dtype = dtype)
sports.head()

,year,unitid,institution_name,city_txt,state_cd,zip_text,classification_code,classification_name,classification_other,ef_male_count,...,partic_coed_women,sum_partic_men,sum_partic_women,rev_men,rev_women,total_rev_menwomen,exp_men,exp_women,total_exp_menwomen,sports
0,2015,100654,Alabama A & M University,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,31,0,345592.0,NaN,345592.0,397818.0,NaN,397818.0,Baseball
1,2015,100654,Alabama A & M University,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,19,16,1211095.0,748833.0,1959928.0,817868.0,742460.0,1560328.0,Basketball
2,2015,100654,Alabama A & M University,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,61,46,183333.0,315574.0,498907.0,246949.0,251184.0,498133.0,All Track Combined
3,2015,100654,Alabama A & M University,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,99,0,2808949.0,NaN,2808949.0,3059353.0,NaN,3059353.0,Football
4,2015,100654,Alabama A & M University,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,9,0,78270.0,NaN,78270.0,83913.0,NaN,83913.0,Golf


This data set contains information about funding and staffing of athletics
programs at U.S. universities from 2015 to 2019 (inclusive). The source for the
data set is the U.S. Department of Education's [Equity in Athletics Data
Analysis project][EADA], and the data was cleaned by the [Tidy Tuesday][tidy]
community. [This article][usafacts] describes some of the conclusions that can
be drawn from the data set.

[EADA]: https://ope.ed.gov/athletics/#/datafile/list
[tidy]: https://github.com/rfordatascience/tidytuesday/tree/master/data/2022/2022-03-29
[usafacts]: https://usafacts.org/articles/coronavirus-college-football-profit-sec-acc-pac-12-big-ten-millions-fall-2020/

On data frames, you can use the `.index` attribute to get and set the *row*
index. For example:

In [12]:
sports.index

RangeIndex(start=0, stop=132327, step=1)

The default index created by the `read_csv` function counts from 0 to the
number of rows.

The other index on data frames corresponds to the columns. You can use the
`.columns` attribute to get and set the column index. For instance:

In [13]:
sports.columns

Index(['year', 'unitid', 'institution_name', 'city_txt', 'state_cd',
       'zip_text', 'classification_code', 'classification_name',
       'classification_other', 'ef_male_count', 'ef_female_count',
       'ef_total_count', 'sector_cd', 'sector_name', 'sportscode',
       'partic_men', 'partic_women', 'partic_coed_men', 'partic_coed_women',
       'sum_partic_men', 'sum_partic_women', 'rev_men', 'rev_women',
       'total_rev_menwomen', 'exp_men', 'exp_women', 'total_exp_menwomen',
       'sports'],
      dtype='object')

The indexes stored in the `.index` and `.columns` attributes of a data frame
have the same classes and basic properties as indexes on series.

(alignment)=
### Alignment

For arithmetic operations that involve more than one series or data frame,
pandas automatically **aligns** the elements based on the indexes. As an
example, consider these two series:

In [14]:
u = pd.Series([1, 2, 3], index = ["a", "b", "c"])
v = pd.Series([1, 2, 3], index = ["c", "b", "a"])

The elements of the series are identical, but the indexes differ. If you're an
experienced NumPy (or R) user, the result of adding these two series together
might surprise you:

In [15]:
u + v

a    4
b    4
c    4
dtype: int64

This is the result because pandas aligns the elements, adding element `a` from
`u` to element `a` from `v`, element `b` to element `b`, and so on. 

Sometimes you might want to do arithmetic on series or data frames without any
alignment. The canonical solution in this case is to use the `.to_numpy` method
to convert one or both of the objects into a NumPy array:

In [16]:
u + v.to_numpy()

a    2
b    4
c    6
dtype: int64

Pandas tries to align elements and carry out arithmetic operations even for
objects with mismatched shapes and labels. Pandas fills elements for which the
result is unknown with missing values. For example:

In [17]:
w = pd.Series([-10, 0], index = ["z", "a"])

u - w

a    1.0
b    NaN
c    NaN
z    NaN
dtype: float64

In this example, only element `a` has two operands; for each other element, the
result is a missing value because only one operand is known.

:::{admonition} Checkpoint Exercise
:class: important

What happens if you use an arithmetic operation on two series or data frames
where one has repeated/duplicated labels in its index?

Hint: construct an example and see what happens!
:::

Pandas does *not* align series and data frames in comparison operations, and
raises an error if they're not already aligned:

In [18]:
u > v

ValueError: Can only compare identically-labeled Series objects

See [this GitHub issue][compare-align] for a developer discussion of whether
this is a helpful feature for preventing bugs or a nuisance.

[compare-align]: https://github.com/pandas-dev/pandas/issues/1134

Finally, the `align` method manually aligns two series or data frames. The
method returns two objects of the same type, but with elements sorted so that
the indexes match. For example:

In [19]:
u.align(v)

(a    1
 b    2
 c    3
 dtype: int64,
 a    3
 b    2
 c    1
 dtype: int64)

For objects with mismatched shapes, the `align` method inserts missing values
to make their shapes and indexes match:

In [20]:
u.align(w)

(a    1.0
 b    2.0
 c    3.0
 z    NaN
 dtype: float64,
 a     0.0
 b     NaN
 c     NaN
 z   -10.0
 dtype: float64)

The `align` method provides a way to align to objects so that they can be
compared:

In [21]:
ua, va = u.align(v)
ua > va

a    False
b    False
c     True
dtype: bool

Moreover, concatenating the two results from the `align` method is the same as
performing a [join][]. The `align` method even provides a parameter `join` to
select the type of join (`"outer"`, `"left"`, `"right"`, or `"inner"`). For
relational series and data frames, you can break a join into smaller steps by
using a combination of the `align` function , column indexing, and the pandas
`concat` function.

[join]: https://pandas.pydata.org/docs/user_guide/merging.html#database-style-dataframe-or-named-series-joining-merging


Indexing Operators
------------------

Pandas provides several different operators for indexing series and data
frames, as well as several different ways to use each. The primary indexing
operator is the square bracket `[ ]`.

For a `Series`, the square bracket operator *usually* selects elements by
label. For instance, integer arguments are treated as labels, not positions, if
the labels in the index are integers:

In [22]:
x = pd.Series([1, 2, 3], index = [1, 0, 3])
x[0]

2

You can use a list to select multiple elements at once or repeat elements:

In [23]:
x[[0, 3, 0]]

0    2
3    3
0    2
dtype: int64

You can also use the square bracket operator to select elements by label when
the labels are strings:

In [24]:
y = pd.Series([10, 20, 30], index = ["a", "b", "c"])
y["a"]

10

In this case, passing an integer argument selects an element by position:

In [25]:
y[0]

10

Using the square bracket operator this way in non-interactive code is risky,
because it can cause bugs if some of the series or data frames your code
operates on unexpectedly have integer labels.

The dot `.` serves as a secondary indexing operator for series and data frames
with string indexes. For instance, to get the element `b`:

In [26]:
y.b

20

Since the dot `.` is also used to access attributes and methods, it cannot be
used to access elements with a label that's the same as the name of an
attribute or method. When in doubt, it's safer to use `[ ]` to access elements
by label.


### Slices

A **slice** selects a range of elements. The syntax is `start:stop:step`, with
the second colon `:` and arguments optional, the same as for Python's built-in
slicing. The default start value is the beginning of the series, the default
stop value is one past the end, and the default step value is 1.

Slices can be by position or by label. With `[ ]`, when the start or stop value
is an integer, pandas assumes you want to slice by position:

In [27]:
x[1:3]

0    2
3    3
dtype: int64

In a slice by position, the element at the stop position is not included.

Slicing with only the step can be useful in a variety of situations, such as
getting every second element:

In [28]:
x[::2]

1    1
3    3
dtype: int64

Negative values in a slice are counted backward from the end of the object. For
example, this code gets the *last* two elements of the series:

In [29]:
x[-2:]

0    2
3    3
dtype: int64

:::{admonition} Checkpoint Exercise
:class: important

What happens if you use a negative step value in a slice?
:::


When the start or stop value is a string, pandas slices by label:

In [30]:
y["b":]

b    20
c    30
dtype: int64

Be aware that when you slice by label, the element at the stop position is
always included:

In [31]:
y["b":"c"]

b    20
c    30
dtype: int64

This is a case where the pandas developers decided convenience outweighs
consistency. See [this section][endpoints] of the pandas documentation for more
details.

[endpoints]: https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html#advanced-endpoints-are-inclusive


:::{admonition} Checkpoint Exercise
:class: important

What happens if you include an integer step value in a slice by label?
:::


### By Position with `.iloc`

When you want to index a series or data frame by position, use `[ ]` on the
`.iloc` attribute. Then any integer arguments are assumed to be positions,
regardless of the type of index on the object.

For example:

In [32]:
x.iloc[0]

1

In [33]:
x.iloc[[1, 0, 2]]

0    2
1    1
3    3
dtype: int64

In [34]:
x.iloc[0:]

1    1
0    2
3    3
dtype: int64

You can use this attribute regardless of the index type:

In [35]:
y.iloc[0:]

a    10
b    20
c    30
dtype: int64

As a result, the `.iloc` attribute is a more reliable way to index by position
than using `[ ]` alone.

Pandas also provides an `.iat` attribute to get or set scalar values in a series
or data frame by position:

In [36]:
x.iat[0]

1

Compared to `.iloc`, the `.iat` attribute is typically much faster (in terms of
CPU time), so it's important to use `.iat` in performance-critical sections of
code. In addition, `.iat` raises an error if your code attempts to get or set
more than one value, which can potentially alert you to bugs that would go
undetected with `.iloc`.


### By Label with `.loc`

When you want to index a series or data frame by label, use `[ ]` on the
`.loc` attribute. Then any integer arguments are assumed to be labels.

For example:

In [37]:
x.loc[0]

2

In [38]:
x.loc[0]

2

In [39]:
x.loc[[0, 1]]

0    2
1    1
dtype: int64

In [40]:
y.loc["b"]

20

In [41]:
y.loc[["b", "a", "b"]]

b    20
a    10
b    20
dtype: int64

Pandas also provides an `.at` attribute to get or set scalar values in a series
or data frame by label:

In [42]:
x.at[0]

2

The advantages of `.at` compared to `.loc` are the same as `.iat` compared to
`.iloc`: better run-time speed and stricter requirements on the argument.

When setting elements by label with `[ ]`, `.loc`, or `.at`, if you use a
label that's not already in the series index, pandas will append the label and
value. For instance:

In [43]:
x.loc[10] = -1
x

1     1
0     2
3     3
10   -1
dtype: int64

This doesn't work when setting elements by position.


### By a Condition

The square bracket operator `[ ]`, the `.iloc` attribute, and the `.loc`
attribute all support indexing a series by a condition. The condition must be
represented by a Boolean array with the same length as the series.

You can use comparison operators to get a suitable Boolean array. For example:

In [44]:
is_positive = x > 0
is_positive

1      True
0      True
3      True
10    False
dtype: bool

In [45]:
x[is_positive]

1    1
0    2
3    3
dtype: int64

In [46]:
x.loc[is_positive]

1    1
0    2
3    3
dtype: int64

If the condition has an index, the square bracket operator and `.loc` operator
will try to align the condition with the series being indexed:

In [47]:
z = x.iloc[[1, 0, 3]]
z

0     2
1     1
10   -1
dtype: int64

In [48]:
z[is_positive]

0    2
1    1
dtype: int64

The `.iloc` operator raises an error if the Boolean array has an index:

In [49]:
x.iloc[is_positive]

NotImplementedError: iLocation based boolean indexing on an integer type is not available

As described in {numref}`alignment`, the canonical way to avoid index alignment
is to use the `.to_numpy` method to convert to a NumPy array:

In [50]:
x.iloc[is_positive.to_numpy()]

1    1
0    2
3    3
dtype: int64

The logic operators for inverting and combining conditions in pandas differ
from Python's usual logic operators:

* `|` is logical or
* `&` is logical and
* `~` is logical not

For example:

In [51]:
x[is_positive & (x % 2 == 1)]

1    1
3    3
dtype: int64

In compound conditions, each sub-condition must be enclosed in parentheses
`()`, because in Python's [order of operations][oo], the `|`, `&`, and `~`
operators are evaluated before comparison operators such as `==`, `>`, and `<`.

[oo]: https://docs.python.org/3/reference/expressions.html#operator-precedence


### By a Callable

The square bracket operator `[ ]`, the `.iloc` attribute, and the `.loc`
attribute also all support indexing a series by a function (or **callable**
object defined with `def` or `lambda`. The function must accept one argument:
the series itself. The function must return a valid argument for the indexing
operator (positions, labels, or a Boolean array).

The primary use case for this form of indexing is chaining together multiple
operations without defining intermediate variables. For example, this code adds
each element at an even-numbered position to the next element, and then gets
only the results greater than 2:

In [52]:
(x.iloc[::2] + x.iloc[1::2].to_numpy())[lambda a: a > 2]

1    3
dtype: int64

As you can probably see, chaining operations makes code relatively difficult to
understand. Whenever possible, it's better to use simple indexing strategies
rather than indexing by a callable.

:::{admonition} Checkpoint Exercise
:class: important

Use indexing by a callable to set the elements of `x` with even values (not
positions) to twice their current value.

What's a simpler way to do this using other indexing modes?
:::


### Data Frames

The indexing operators `[ ]`, `.iloc`, and `.loc` can all be used with data
frames in addition to series. Since data frames are 2-dimensional, you can pass
the indexing operators a separate argument for each dimension. The rows come
first.

For instance, to get (1st row, 2nd and 3rd columns) of the athletics data frame
the code is:

In [53]:
sports.iloc[0, [1, 2]]

unitid                                100654
institution_name    Alabama A & M University
Name: 0, dtype: object

When the result has a lower dimension than the object being indexed, pandas
automatically converts to an appropriate data type. In this case, the result is
converted to a series with the names of the two columns as its index.

As another example, the code to get (rows labeled 10 and 11, column labeled
`"year"`) is:

In [54]:
sports.loc[[10, 11], "year"]

10    2015
11    2015
Name: year, dtype: int64

With `.iloc` and `.loc`, you can select all of the elements along a given
dimension by passing a colon `:` as the argument for that dimension. For
instance, this code gets all rows in the columns labeled `"year"` and
`"institution_name"`:

In [55]:
sports.loc[:, ["year", "institution_name"]]

,year,institution_name
0,2015,Alabama A & M University
1,2015,Alabama A & M University
2,2015,Alabama A & M University
3,2015,Alabama A & M University
4,2015,Alabama A & M University
...,...,...
132322,2019,Simon Fraser University
132323,2019,Simon Fraser University
132324,2019,Simon Fraser University
132325,2019,Simon Fraser University


You can mix indexing by condition with indexing by position or label. The most
common way to do this is to use a condition to select rows and labels to select
columns. For example:

In [56]:
sports.loc[sports.year == 2018, ["year", "institution_name", "sports"]]

,year,institution_name,sports
52387,2018,Alabama A & M University,Baseball
52388,2018,Alabama A & M University,Basketball
52389,2018,Alabama A & M University,Football
52390,2018,Alabama A & M University,Golf
52391,2018,Alabama A & M University,Softball
...,...,...,...
70154,2018,Simon Fraser University,Soccer
70155,2018,Simon Fraser University,Softball
70156,2018,Simon Fraser University,Swimming
70157,2018,Simon Fraser University,Volleyball


:::{admonition} Checkpoint Exercise
:class: important

Use indexing to get only the rows where `year` is `2016` and `institution_name`
is `"University of California-Davis"`. Limit the columns to `year`,
`institution_name`, `sports`, and `total_rev_menwomen` (the total revenue from
the program in USD).
:::


Indexing a data frame with only one argument generally accesses rows. For
example:

In [57]:
sports.iloc[0]

year                                        2015
unitid                                    100654
institution_name        Alabama A & M University
city_txt                                  Normal
state_cd                                      AL
zip_text                                 35762.0
classification_code                            2
classification_name          NCAA Division I-FCS
classification_other                         NaN
ef_male_count                               1923
ef_female_count                             2300
ef_total_count                              4223
sector_cd                                      1
sector_name              Public, 4-year or above
sportscode                                     1
partic_men                                  31.0
partic_women                                 NaN
partic_coed_men                              NaN
partic_coed_women                            NaN
sum_partic_men                                31
sum_partic_women    

As an exception, using the square bracket operator `[ ]` to index a data frame
with a string argument (or list of strings) accesses columns:

In [58]:
sports[["year", "institution_name"]]

,year,institution_name
0,2015,Alabama A & M University
1,2015,Alabama A & M University
2,2015,Alabama A & M University
3,2015,Alabama A & M University
4,2015,Alabama A & M University
...,...,...
132322,2019,Simon Fraser University
132323,2019,Simon Fraser University
132324,2019,Simon Fraser University
132325,2019,Simon Fraser University


(column-indexes)=
Columns as Indexes
------------------

An important feature of pandas is that you can use arbitrary columns in a data
frame as indexes. Generally the columns should contain identifiers for the rows
or other categorical data. For example, in the `sports` data frame, the column
`institution_name` is an identifier.

You can use the `.set_index` method to set a column as an index:

In [59]:
sports = sports.set_index("institution_name")
sports.head()

,year,unitid,city_txt,state_cd,zip_text,classification_code,classification_name,classification_other,ef_male_count,ef_female_count,...,partic_coed_women,sum_partic_men,sum_partic_women,rev_men,rev_women,total_rev_menwomen,exp_men,exp_women,total_exp_menwomen,sports
institution_name,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,2300,...,NaN,31,0,345592.0,NaN,345592.0,397818.0,NaN,397818.0,Baseball
Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,2300,...,NaN,19,16,1211095.0,748833.0,1959928.0,817868.0,742460.0,1560328.0,Basketball
Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,2300,...,NaN,61,46,183333.0,315574.0,498907.0,246949.0,251184.0,498133.0,All Track Combined
Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,2300,...,NaN,99,0,2808949.0,NaN,2808949.0,3059353.0,NaN,3059353.0,Football
Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,2300,...,NaN,9,0,78270.0,NaN,78270.0,83913.0,NaN,83913.0,Golf


After setting a categorical column as an index, you can use indexing by label
as a convenient way to get rows in specific categories:

In [60]:
sports.loc["University of California-Davis"]

,year,unitid,city_txt,state_cd,zip_text,classification_code,classification_name,classification_other,ef_male_count,ef_female_count,...,partic_coed_women,sum_partic_men,sum_partic_women,rev_men,rev_women,total_rev_menwomen,exp_men,exp_women,total_exp_menwomen,sports
institution_name,,,,,,,,,,,,,,,,,,,,,
University of California-Davis,2015,110644,Davis,CA,956168678.0,2,NCAA Division I-FCS,NaN,11215,16223,...,NaN,36,0,884627.0,NaN,884627.0,884627.0,NaN,884627.0,Baseball
University of California-Davis,2015,110644,Davis,CA,956168678.0,2,NCAA Division I-FCS,NaN,11215,16223,...,NaN,15,19,1862783.0,1340541.0,3203324.0,1862783.0,1340541.0,3203324.0,Basketball
University of California-Davis,2015,110644,Davis,CA,956168678.0,2,NCAA Division I-FCS,NaN,11215,16223,...,NaN,62,94,571582.0,590416.0,1161998.0,571582.0,590416.0,1161998.0,All Track Combined
University of California-Davis,2015,110644,Davis,CA,956168678.0,2,NCAA Division I-FCS,NaN,11215,16223,...,NaN,0,25,NaN,659996.0,659996.0,NaN,659996.0,659996.0,Field Hockey
University of California-Davis,2015,110644,Davis,CA,956168678.0,2,NCAA Division I-FCS,NaN,11215,16223,...,NaN,105,0,4478087.0,NaN,4478087.0,4478087.0,NaN,4478087.0,Football
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
University of California-Davis,2019,110644,Davis,CA,95616.0,2,NCAA Division I-FCS,NaN,11780,18371,...,NaN,0,27,NaN,945919.0,945919.0,NaN,945919.0,945919.0,Equestrian
University of California-Davis,2019,110644,Davis,CA,95616.0,2,NCAA Division I-FCS,NaN,11780,18371,...,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Rodeo
University of California-Davis,2019,110644,Davis,CA,95616.0,2,NCAA Division I-FCS,NaN,11780,18371,...,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Sailing


You can use the `.reset_index` method to restore the `institution_name` column
to the data frame and reset the index to a range:

In [61]:
sports = sports.reset_index()
sports.head()

,institution_name,year,unitid,city_txt,state_cd,zip_text,classification_code,classification_name,classification_other,ef_male_count,...,partic_coed_women,sum_partic_men,sum_partic_women,rev_men,rev_women,total_rev_menwomen,exp_men,exp_women,total_exp_menwomen,sports
0,Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,31,0,345592.0,NaN,345592.0,397818.0,NaN,397818.0,Baseball
1,Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,19,16,1211095.0,748833.0,1959928.0,817868.0,742460.0,1560328.0,Basketball
2,Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,61,46,183333.0,315574.0,498907.0,246949.0,251184.0,498133.0,All Track Combined
3,Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,99,0,2808949.0,NaN,2808949.0,3059353.0,NaN,3059353.0,Football
4,Alabama A & M University,2015,100654,Normal,AL,35762.0,2,NCAA Division I-FCS,NaN,1923,...,NaN,9,0,78270.0,NaN,78270.0,83913.0,NaN,83913.0,Golf


Setting indexes appropriately makes it much more convenient to filter the rows
of a data frame.


### MultiIndexes

One of pandas' most powerful features is the ability to set multiple columns as
the index at the same time. This creates a `MultiIndex`, a hierarchical index
with multiple levels. For example, this code creates a MultiIndex from `year`,
`institution_name` and `sports`:

In [62]:
sports = sports.set_index(["year", "institution_name", "sports"])

The MultiIndex makes it easy to get all of the information for a specific year,
institution, and sport. When working with a MultiIndex, the corresponding
argument to the indexing operator should be a tuple with one element for each
level of the MultiIndex:

In [63]:
sports.loc[(2015, "Stanford University", "Football")]

/tmp/ipykernel_77509/1887213640.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  sports.loc[(2015, "Stanford University", "Football")]


,,,unitid,city_txt,state_cd,zip_text,classification_code,classification_name,classification_other,ef_male_count,ef_female_count,ef_total_count,...,partic_coed_men,partic_coed_women,sum_partic_men,sum_partic_women,rev_men,rev_women,total_rev_menwomen,exp_men,exp_women,total_exp_menwomen
year,institution_name,sports,,,,,,,,,,,,,,,,,,,,,
2015,Stanford University,Football,243744,Stanford,CA,94305.0,1,NCAA Division I-FBS,NaN,3663,3331,6994,...,NaN,NaN,100,0,43744639.0,NaN,43744639.0,23724407.0,NaN,23724407.0


You can use the `.xs` (read: "cross section") method to select rows from only
one particular level of a MultiIndex. For instance, to get all rows for Harvard
university:

In [64]:
sports.xs("Harvard University", level = 1)

unitid   city_txt state_cd  zip_text  \
year sports                                                     
2015 Baseball            166027  Cambridge       MA    2138.0   
     Basketball          166027  Cambridge       MA    2138.0   
     All Track Combined  166027  Cambridge       MA    2138.0   
     Fencing             166027  Cambridge       MA    2138.0   
     Field Hockey        166027  Cambridge       MA    2138.0   
...                         ...        ...      ...       ...   
2019 Equestrian          166027  Cambridge       MA    2138.0   
     Rodeo               166027  Cambridge       MA    2138.0   
     Sailing             166027  Cambridge       MA    2138.0   
     Table Tennis        166027  Cambridge       MA    2138.0   
     Weight Lifting      166027  Cambridge       MA    2138.0   

                         classification_code  classification_name  \
year sports                                                         
2015 Baseball                              2  NCAA Division I-FCS   
     Basketball                            2  NCAA Division I-FCS   
     All Track Combined                    2  NCAA Division I-FCS   
     Fencing                               2  NCAA Division I-FCS   
     Field Hockey                          2  NCAA Division I-FCS   
...                                      ...                  ...   
2019 Equestrian                            2  NCAA Division I-FCS   
     Rodeo                                 2  NCAA Division I-FCS   
     Sailing                               2  NCAA Division I-FCS   
     Table Tennis                          2  NCAA Division I-FCS   
     Weight Lifting                        2  NCAA Division I-FCS   

                        classification_other  ef_male_count  ef_female_count  \
year sports                                                                    
2015 Baseball                            NaN           3637             3256   
     Basketball                          NaN           3637             3256   
     All Track Combined                  NaN           3637             3256   
     Fencing                             NaN           3637             3256   
     Field Hockey                        NaN           3637             3256   
...                                      ...            ...              ...   
2019 Equestrian                          NaN           3531             3454   
     Rodeo                               NaN           3531             3454   
     Sailing                             NaN           3531             3454   
     Table Tennis                        NaN           3531             3454   
     Weight Lifting                      NaN           3531             3454   

                         ef_total_count  ...  partic_coed_men  \
year sports                              ...                    
2015 Baseball                      6893  ...              NaN   
     Basketball                    6893  ...              NaN   
     All Track Combined            6893  ...              NaN   
     Fencing                       6893  ...              NaN   
     Field Hockey                  6893  ...              NaN   
...                                 ...  ...              ...   
2019 Equestrian                    6985  ...              NaN   
     Rodeo                         6985  ...              NaN   
     Sailing                       6985  ...              NaN   
     Table Tennis                  6985  ...              NaN   
     Weight Lifting                6985  ...              NaN   

                        partic_coed_women  sum_partic_men  sum_partic_women  \
year sports                                                                   
2015 Baseball                         NaN              35                 0   
     Basketball                       NaN              16                19   
     All Track Combined               NaN             138                95   

The level argument is `1` because `institution_name` is the 2nd level of the
MultiIndex, and indexing starts from 0. You can also use the name of the level
instead:

In [65]:
sports.xs("Princeton University", level = "institution_name")

unitid   city_txt state_cd    zip_text  \
year sports                                                        
2015 Baseball             186131  Princeton       NJ  85440070.0   
     Basketball           186131  Princeton       NJ  85440070.0   
     All Track Combined   186131  Princeton       NJ  85440070.0   
     Fencing              186131  Princeton       NJ  85440070.0   
     Field Hockey         186131  Princeton       NJ  85440070.0   
...                          ...        ...      ...         ...   
2019 Swimming and Diving  186131  Princeton       NJ      8544.0   
     Tennis               186131  Princeton       NJ      8544.0   
     Volleyball           186131  Princeton       NJ      8544.0   
     Water Polo           186131  Princeton       NJ      8544.0   
     Wrestling            186131  Princeton       NJ      8544.0   

                          classification_code  classification_name  \
year sports                                                          
2015 Baseball                               2  NCAA Division I-FCS   
     Basketball                             2  NCAA Division I-FCS   
     All Track Combined                     2  NCAA Division I-FCS   
     Fencing                                2  NCAA Division I-FCS   
     Field Hockey                           2  NCAA Division I-FCS   
...                                       ...                  ...   
2019 Swimming and Diving                    2  NCAA Division I-FCS   
     Tennis                                 2  NCAA Division I-FCS   
     Volleyball                             2  NCAA Division I-FCS   
     Water Polo                             2  NCAA Division I-FCS   
     Wrestling                              2  NCAA Division I-FCS   

                         classification_other  ef_male_count  ef_female_count  \
year sports                                                                     
2015 Baseball                             NaN           2717             2543   
     Basketball                           NaN           2717             2543   
     All Track Combined                   NaN           2717             2543   
     Fencing                              NaN           2717             2543   
     Field Hockey                         NaN           2717             2543   
...                                       ...            ...              ...   
2019 Swimming and Diving                  NaN           2670             2638   
     Tennis                               NaN           2670             2638   
     Volleyball                           NaN           2670             2638   
     Water Polo                           NaN           2670             2638   
     Wrestling                            NaN           2670             2638   

                          ef_total_count  ...  partic_coed_men  \
year sports                               ...                    
2015 Baseball                       5260  ...              NaN   
     Basketball                     5260  ...              NaN   
     All Track Combined             5260  ...              NaN   
     Fencing                        5260  ...              NaN   
     Field Hockey                   5260  ...              NaN   
...                                  ...  ...              ...   
2019 Swimming and Diving            5308  ...              NaN   
     Tennis                         5308  ...              NaN   
     Volleyball                     5308  ...              NaN   
     Water Polo                     5308  ...              NaN   
     Wrestling                      5308  ...              NaN   

                         partic_coed_women  sum_partic_men  sum_partic_women  \
year sports                                                                    
2015 Baseball                          NaN              31                 0   
     Basketball                        NaN              16                17 

The [pandas documentation on MultiIndexes][multiindex-docs] describes a wide
variety of ways MultiIndexes can be used to slice and dice data frames.

[multiindex-docs]: https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html

:::{admonition} Checkpoint Exercise
:class: important

Use a MultiIndex to get a series that lists the sports for which there were
university athletics programs in Aberdeen, WA in 2018.
:::


The `.query` Method
-----------------

The `.query` method is another powerful way to extract information from a data
frame. The idea of `.query` is that you write a query string using a query
mini-language to select a subset of the rows. This has several advantages over
other ways of indexing:

* For large data sets, the `.query` method is [slightly faster][query-perf]
  than other ways of indexing
* You don't have to repeat the name of the data frame every time you refer to a
  column in a condition
* You can reuse a single query across multiple data frames, provided they all
  have the queried columns
* Conditions are easier to read:
    + You can use `not`, `or` and `and` to invert and combine conditions
      instead of `~`, `|`, and `&`
    * You can use `in` and `not in` in conditions
    * You can write `a < b < c` rather than `(a < b) & (b < c)`
    + It's not necessary to enclose sub-conditions in parentheses, since the
      query mini-language defines its own order of operations

[query-perf]: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#performance-of-query

Here's an example of a query:

In [66]:
sports = sports.reset_index()

qs = "city_txt == 'Davis' and state_cd == 'CA' and sports == 'Basketball'"
ucd_bball = sports.query(qs)
ucd_bball[["institution_name", "sports", "year", "total_rev_menwomen"]]

,institution_name,sports,year,total_rev_menwomen
913,University of California-Davis,Basketball,2015,3203324.0
18263,University of California-Davis,Basketball,2016,3817436.0
35698,University of California-Davis,Basketball,2017,4021500.0
53319,University of California-Davis,Basketball,2018,4240907.0
73947,University of California-Davis,Basketball,2019,4141401.0


:::{admonition} Checkpoint Exercise
:class: important

Use a query string to get a data frame that lists the sports for which there
were university athletics programs in Aberdeen, WA in 2018.
:::

The pandas documentation provides [further details about how to write query
strings][query-strings].

[query-strings]: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#the-query-method